In [0]:
df = spark.read.option("header", "true") \
               .option("multiLine", "true") \
               .option("escape", "\"") \
               .option("quote", "\"") \
               .csv('/Volumes/workspace/default/dataanalysis/data_analysis.csv')
df.show(2)

+------------+---+------------+-----------+---------+--------------------+--------------------+--------------------+----------------------+--------------------+
|Customer ID |Age|        Name|Middle name|Last name|        work address|         Transaction|Transaction end date|Transaction start date|              Review|
+------------+---+------------+-----------+---------+--------------------+--------------------+--------------------+----------------------+--------------------+
|         100|M24|James/Canada|      David|   Harris|{ "street": "12 M...|[$2000, $3000, $3...|    10-01-2025 01:16|      10-01-2025 00:16|The iPhone's came...|
|         101|F27|    Sham/USA|          P| Anderson| {"street": "210 ...|[$1000, $3500, $4...|    10-01-2025 13:14|      10-01-2025 12:14|The iPhone's Face...|
+------------+---+------------+-----------+---------+--------------------+--------------------+--------------------+----------------------+--------------------+
only showing top 2 rows


In [0]:
df .printSchema()

root
 |-- Customer ID : string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Middle name: string (nullable = true)
 |-- Last name: string (nullable = true)
 |-- work address: string (nullable = true)
 |-- Transaction: string (nullable = true)
 |-- Transaction end date: string (nullable = true)
 |-- Transaction start date: string (nullable = true)
 |-- Review: string (nullable = true)



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
address_schema = StructType([
    StructField("street", StringType(), True),
    StructField("city", StringType(), True)
])
df_parsed = df.withColumn("address", from_json(col("work address"), address_schema))
df_final = df_parsed \
    .withColumn("street", col("address.street")) \
    .withColumn("city", col("address.city")) \
    .drop("address")
df_json=df_final.select('work address','street','city')
df_json.show(2)

+--------------------+-----------------+-----------+
|        work address|           street|       city|
+--------------------+-----------------+-----------+
|{ "street": "12 M...|  12 Maple Street|    Toronto|
| {"street": "210 ...|210 Lakeview Blvd|Mississauga|
+--------------------+-----------------+-----------+
only showing top 2 rows


In [0]:
df_1= df_final.withColumn('Transaction', regexp_replace(col('Transaction'),r'\$|\[|\]',''))\
              .withColumn('Transaction1', try_element_at(split(col('Transaction'),','), lit(1)))\
              .withColumn('Transaction2', try_element_at(split(col('Transaction'),','), lit(2)))\
              .withColumn('Transaction3', try_element_at(split(col('Transaction'),','), lit(3)))\
              .withColumn('Transaction1', col('Transaction1').cast('int'))\
              .withColumn('Transaction2', col('Transaction2').cast('int'))\
              .withColumn('Transaction3', col('Transaction3').cast('int'))
df_list= df_1.select('Transaction','Transaction1','Transaction2','Transaction3')
df_list.show(2)

+----------------+------------+------------+------------+
|     Transaction|Transaction1|Transaction2|Transaction3|
+----------------+------------+------------+------------+
|2000, 3000, 3500|        2000|        3000|        3500|
|1000, 3500, 4500|        1000|        3500|        4500|
+----------------+------------+------------+------------+
only showing top 2 rows


In [0]:
df_1= df_1.withColumn(
    "total_amount",
    coalesce(col("Transaction1"), lit(0)) +
    coalesce(col("Transaction2"), lit(0)) +
    coalesce(col("Transaction3"), lit(0)))\
         .withColumn('Firstname', try_element_at(split(col('Name'), '/'), lit(1)))\
         .withColumn('Location', try_element_at(split(col('Name'), '/'), lit(2)))\
         .withColumn('Fullname',concat(col('Firstname'), col('Middle name'),col('Last name')))\
         .withColumn('Category', regexp_extract(col("Review"), r'(?i)(iPhone|Samsung Galaxy|Motorola)',1))\
         .withColumn("Sex",  regexp_extract(col('Age'),r'(M|F)',1))\
         .withColumn('Age',substring(col('Age'),2,3))\
         .drop('Transaction1','Transaction2','Transaction3','Transaction','Name','Middle name','Last name','Firstname','Review','work address')
df_1.show(2)

+------------+---+--------------------+----------------------+-----------------+-----------+------------+--------+----------------+--------+---+
|Customer ID |Age|Transaction end date|Transaction start date|           street|       city|total_amount|Location|        Fullname|Category|Sex|
+------------+---+--------------------+----------------------+-----------------+-----------+------------+--------+----------------+--------+---+
|         100| 24|    10-01-2025 01:16|      10-01-2025 00:16|  12 Maple Street|    Toronto|        8500|  Canada|JamesDavidHarris|  iPhone|  M|
|         101| 27|    10-01-2025 13:14|      10-01-2025 12:14|210 Lakeview Blvd|Mississauga|        9000|     USA|   ShamPAnderson|  iPhone|  F|
+------------+---+--------------------+----------------------+-----------------+-----------+------------+--------+----------------+--------+---+
only showing top 2 rows


In [0]:
df_2=df_1.drop('Fullname','street','city')

In [0]:
df_2.write.mode('overwrite').format('csv').option("header",True).save('/Volumes/workspace/default/dataanalysis/data_analysis_cleaned.csv')

In [0]:
df= spark.read.csv('/Volumes/workspace/default/dataanalysis/data_analysis_cleaned.csv', inferSchema=True, header=True)
df.show()

+-----------+----+--------------------+----------------------+------------+--------+--------------+----+
|Customer ID| Age|Transaction end date|Transaction start date|total_amount|Location|      Category| Sex|
+-----------+----+--------------------+----------------------+------------+--------+--------------+----+
|        100|  24|    10-01-2025 01:16|      10-01-2025 00:16|        8500|  Canada|        iPhone|   M|
|        101|  27|    10-01-2025 13:14|      10-01-2025 12:14|        9000|     USA|        iPhone|   F|
|        102|  30|    10-01-2025 13:00|      10-01-2025 00:00|       11000|     USA|Samsung Galaxy|   M|
|        103|  33|    21-02-2025 12:35|      21-02-2025 00:00|        8500|     USA|      Motorola|   F|
|        104| 567|    21-02-2025 13:45|      21-02-2025 00:00|        9000|  Canada|        iPhone|NULL|
|        105|NULL|    21-02-2025 00:12|      21-02-2025 00:00|       11000|     USA|Samsung Galaxy|NULL|
|        106|  42|    21-02-2025 00:08|      21-02-2025

In [0]:
df_mean = df.agg(mean('total_amount').alias('amount'))
df_mean.show()

+-------+
| amount|
+-------+
|9843.75|
+-------+



In [0]:
df.createOrReplaceTempView("dfm")

df_missing = spark.sql("""
SELECT * FROM dfm
WHERE `Customer ID` IS NULL OR
Age IS NULL OR 
`Transaction end date` is NULL OR
`Transaction start date` is NULL OR
total_amount is NULL OR
Location is NULL OR
Category is NULL OR
Sex is NULL
""")
df_missing.show()

+-----------+----+--------------------+----------------------+------------+--------+--------------+----+
|Customer ID| Age|Transaction end date|Transaction start date|total_amount|Location|      Category| Sex|
+-----------+----+--------------------+----------------------+------------+--------+--------------+----+
|        104| 567|    21-02-2025 13:45|      21-02-2025 00:00|        9000|  Canada|        iPhone|NULL|
|        105|NULL|    21-02-2025 00:12|      21-02-2025 00:00|       11000|     USA|Samsung Galaxy|NULL|
|        113|   6|    05-03-2025 00:03|      05-03-2025 00:00|       13500|     USA|      Motorola|NULL|
|        114|NULL|    05-03-2025 00:10|      05-03-2025 00:00|       11000|  Canada|      Motorola|   M|
|        115|NULL|    21-02-2025 00:30|      21-02-2025 00:00|        8500|   India|Samsung Galaxy|   F|
+-----------+----+--------------------+----------------------+------------+--------+--------------+----+



In [0]:
df_median= df.approxQuantile('Age', [0.5],0)[0]
df_mode= df.groupBy('Sex').count().orderBy(col('count').desc()).first()[0]
df=df.fillna({'Age':df_median, 'Sex':df_mode})
df.show(2)

+-----------+---+--------------------+----------------------+------------+--------+--------+---+
|Customer ID|Age|Transaction end date|Transaction start date|total_amount|Location|Category|Sex|
+-----------+---+--------------------+----------------------+------------+--------+--------+---+
|        100| 24|    10-01-2025 01:16|      10-01-2025 00:16|        8500|  Canada|  iPhone|  M|
|        101| 27|    10-01-2025 13:14|      10-01-2025 12:14|        9000|     USA|  iPhone|  F|
+-----------+---+--------------------+----------------------+------------+--------+--------+---+
only showing top 2 rows


In [0]:
Q1= df.approxQuantile('Age', [0.25],0)[0]
Q3= df.approxQuantile('Age', [0.75],0)[0]
IQR= Q3-Q1
lower_limit= Q1 - 1.5* IQR
upper_limit= Q3 + 1.5* IQR
df= df.withColumn('Age', when(col('Age')>upper_limit, upper_limit)
                        .when(col('Age')<lower_limit, lower_limit)
                        .otherwise(col('Age'))                
                  )
df.show()

+-----------+----+--------------------+----------------------+------------+--------+--------------+---+
|Customer ID| Age|Transaction end date|Transaction start date|total_amount|Location|      Category|Sex|
+-----------+----+--------------------+----------------------+------------+--------+--------------+---+
|        100|25.5|    10-01-2025 01:16|      10-01-2025 00:16|        8500|  Canada|        iPhone|  M|
|        101|27.0|    10-01-2025 13:14|      10-01-2025 12:14|        9000|     USA|        iPhone|  F|
|        102|30.0|    10-01-2025 13:00|      10-01-2025 00:00|       11000|     USA|Samsung Galaxy|  M|
|        103|33.0|    21-02-2025 12:35|      21-02-2025 00:00|        8500|     USA|      Motorola|  F|
|        104|37.5|    21-02-2025 13:45|      21-02-2025 00:00|        9000|  Canada|        iPhone|  F|
|        105|33.0|    21-02-2025 00:12|      21-02-2025 00:00|       11000|     USA|Samsung Galaxy|  F|
|        106|37.5|    21-02-2025 00:08|      21-02-2025 00:00|  

In [0]:
df_pivot= df.groupBy('Location').pivot('Category').sum('total_amount')
df_pivot.show()

+--------+--------+--------------+------+
|Location|Motorola|Samsung Galaxy|iPhone|
+--------+--------+--------------+------+
|     USA|   41000|         22000|  9000|
|  Canada|   19500|         22000| 17500|
|   India|    5000|          8500| 13000|
+--------+--------+--------------+------+



In [0]:
df_stack = df_pivot.select('Location', expr("stack(3, 'Motorola', Motorola, 'Samsung Galaxy', `Samsung Galaxy`, 'iPhone', iPhone) as (Category, Amount)"))
df_stack.show()

+--------+--------------+------+
|Location|      Category|Amount|
+--------+--------------+------+
|     USA|      Motorola| 41000|
|     USA|Samsung Galaxy| 22000|
|     USA|        iPhone|  9000|
|  Canada|      Motorola| 19500|
|  Canada|Samsung Galaxy| 22000|
|  Canada|        iPhone| 17500|
|   India|      Motorola|  5000|
|   India|Samsung Galaxy|  8500|
|   India|        iPhone| 13000|
+--------+--------------+------+



In [0]:
from pyspark.sql.functions import when
df= df.withColumn('customer_category', 
                      when((col('total_amount')>10000) & (col('Location')== 'USA'), 'High')\
                    .when(col('total_amount')>8000, 'Medium')\
                    .otherwise('Low'))
df.show()

+-----------+----+--------------------+----------------------+------------+--------+--------------+---+-----------------+
|Customer ID| Age|Transaction end date|Transaction start date|total_amount|Location|      Category|Sex|customer_category|
+-----------+----+--------------------+----------------------+------------+--------+--------------+---+-----------------+
|        100|25.5|    10-01-2025 01:16|      10-01-2025 00:16|        8500|  Canada|        iPhone|  M|           Medium|
|        101|27.0|    10-01-2025 13:14|      10-01-2025 12:14|        9000|     USA|        iPhone|  F|           Medium|
|        102|30.0|    10-01-2025 13:00|      10-01-2025 00:00|       11000|     USA|Samsung Galaxy|  M|             High|
|        103|33.0|    21-02-2025 12:35|      21-02-2025 00:00|        8500|     USA|      Motorola|  F|           Medium|
|        104|37.5|    21-02-2025 13:45|      21-02-2025 00:00|        9000|  Canada|        iPhone|  F|           Medium|
|        105|33.0|    21

In [0]:
df= df.withColumn('Transaction end date', expr('to_timestamp(`Transaction end date`,"dd-MM-yyyy HH:mm")'))\
          .withColumn('Transaction start date', expr('to_timestamp(`Transaction start date`,"dd-MM-yyyy HH:mm")'))\
          .withColumn('timediff', expr('datediff(Minute,`Transaction start date`,`Transaction end date`)'))
df.show(2)

+-----------+----+--------------------+----------------------+------------+--------+--------+---+-----------------+--------+
|Customer ID| Age|Transaction end date|Transaction start date|total_amount|Location|Category|Sex|customer_category|timediff|
+-----------+----+--------------------+----------------------+------------+--------+--------+---+-----------------+--------+
|        100|25.5| 2025-01-10 01:16:00|   2025-01-10 00:16:00|        8500|  Canada|  iPhone|  M|           Medium|      60|
|        101|27.0| 2025-01-10 13:14:00|   2025-01-10 12:14:00|        9000|     USA|  iPhone|  F|           Medium|      60|
+-----------+----+--------------------+----------------------+------------+--------+--------+---+-----------------+--------+
only showing top 2 rows


In [0]:
df_group = df.groupby('Location').agg(
    count('Sex').alias('count_sex'),
    mean('total_amount').alias('Avg_tran')
)
df_group.show()

+--------+---------+------------------+
|Location|count_sex|          Avg_tran|
+--------+---------+------------------+
|     USA|        7|10285.714285714286|
|  Canada|        6| 9833.333333333334|
|   India|        3| 8833.333333333334|
+--------+---------+------------------+



In [0]:
from pyspark.sql.functions import sum
df_group= df_2.groupBy('Location','Category').agg(sum('total_amount').alias('Amount'))
df_rank= df_group.withColumn('rank_id', expr('dense_rank()over(partition by Location order by Amount desc)'))
df_top2= df_rank.where(col('rank_id').isin(1,2))              
df_top2.show()

+--------+--------------+------+-------+
|Location|      Category|Amount|rank_id|
+--------+--------------+------+-------+
|  Canada|Samsung Galaxy| 22000|      1|
|  Canada|      Motorola| 19500|      2|
|   India|        iPhone| 13000|      1|
|   India|Samsung Galaxy|  8500|      2|
|     USA|      Motorola| 41000|      1|
|     USA|Samsung Galaxy| 22000|      2|
+--------+--------------+------+-------+



In [0]:
from pyspark.sql.functions import sum
df_group= df_2.groupBy('Category','Location').agg(sum('total_amount').alias('Amount'))
df_sum= df_group.withColumn('running_total', expr('sum(Amount)over(partition by Category order by Amount desc\
                                                   rows between unbounded preceding and current row )'))
df_avg= df_sum.withColumn('running_average', expr('AVG(Amount)over(partition by Category order by Amount desc\
                                                   rows between 1 preceding and current row )'))                                                   
df_avg.show()

+--------------+--------+------+-------------+---------------+
|      Category|Location|Amount|running_total|running_average|
+--------------+--------+------+-------------+---------------+
|      Motorola|     USA| 41000|        41000|        41000.0|
|      Motorola|  Canada| 19500|        60500|        30250.0|
|      Motorola|   India|  5000|        65500|        12250.0|
|Samsung Galaxy|  Canada| 22000|        22000|        22000.0|
|Samsung Galaxy|     USA| 22000|        44000|        22000.0|
|Samsung Galaxy|   India|  8500|        52500|        15250.0|
|        iPhone|  Canada| 17500|        17500|        17500.0|
|        iPhone|   India| 13000|        30500|        15250.0|
|        iPhone|     USA|  9000|        39500|        11000.0|
+--------------+--------+------+-------------+---------------+



In [0]:
data1 = [
    {"Location": "USA", "City": "Brampton"},
    {"Location": "USA", "City": "Mississauga"}
]
df1 = spark.createDataFrame(data1)
data2 = [
    {"Category": "Samsung Galaxy", "Total": 11000},
    {"Category": "iPhone", "Total": 9000}
]
df2 = spark.createDataFrame(data2)

df1= df1.withColumn('slNo', expr('row_number()over(order by (select NULL))'))
df2= df2.withColumn('slNo', expr('row_number()over(order by (select NULL))'))
df_join= df1.join(df2, df1.slNo==df2.slNo).drop('slNo')
df_join.show()

+-----------+--------+--------------+-----+
|       City|Location|      Category|Total|
+-----------+--------+--------------+-----+
|   Brampton|     USA|Samsung Galaxy|11000|
|Mississauga|     USA|        iPhone| 9000|
+-----------+--------+--------------+-----+



In [0]:
df_union= df1.union(df1)
df_union.show()

+-----------+--------+----+
|       City|Location|slNo|
+-----------+--------+----+
|   Brampton|     USA|   1|
|Mississauga|     USA|   2|
|   Brampton|     USA|   1|
|Mississauga|     USA|   2|
+-----------+--------+----+



In [0]:
df_jason= spark.read.json('/Volumes/workspace/default/dataanalysis/drivers.json')               
df_jason.show(2)

+----+----------+--------+---------+-----------------+-----------+------+--------------------+
|code|       dob|driverId|driverRef|             name|nationality|number|                 url|
+----+----------+--------+---------+-----------------+-----------+------+--------------------+
| HAM|1985-01-07|       1| hamilton|{Lewis, Hamilton}|    British|    44|http://en.wikiped...|
| HEI|1977-05-10|       2| heidfeld| {Nick, Heidfeld}|     German|    \N|http://en.wikiped...|
+----+----------+--------+---------+-----------------+-----------+------+--------------------+
only showing top 2 rows


In [0]:
df_json=df_jason.withColumn('forename', col('name').forename)\
                 .withColumn('surname', col('name').surname)\
                 .drop('name','url')
df_json.show(2)

+----+----------+--------+---------+-----------+------+--------+--------+
|code|       dob|driverId|driverRef|nationality|number|forename| surname|
+----+----------+--------+---------+-----------+------+--------+--------+
| HAM|1985-01-07|       1| hamilton|    British|    44|   Lewis|Hamilton|
| HEI|1977-05-10|       2| heidfeld|     German|    \N|    Nick|Heidfeld|
+----+----------+--------+---------+-----------+------+--------+--------+
only showing top 2 rows


In [0]:
df_jason.printSchema()

root
 |-- code: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- driverId: long (nullable = true)
 |-- driverRef: string (nullable = true)
 |-- name: struct (nullable = true)
 |    |-- forename: string (nullable = true)
 |    |-- surname: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- number: string (nullable = true)
 |-- url: string (nullable = true)
 |-- forename: string (nullable = true)
 |-- surname: string (nullable = true)



In [0]:
%sql
create database sales

In [0]:
%sql
use sales

In [0]:
df_json.write.mode('overwrite').format('delta').saveAsTable('sales.drivers_data')

In [0]:
%sql
select * from sales.drivers_data

code,dob,driverId,driverRef,nationality,number,forename,surname
HAM,1985-01-07,1,hamilton,British,44,Lewis,Hamilton
HEI,1977-05-10,2,heidfeld,German,\N,Nick,Heidfeld
ROS,1985-06-27,3,rosberg,German,6,Nico,Rosberg
ALO,1981-07-29,4,alonso,Spanish,14,Fernando,Alonso
KOV,1981-10-19,5,kovalainen,Finnish,\N,Heikki,Kovalainen
NAK,1985-01-11,6,nakajima,Japanese,\N,Kazuki,Nakajima
BOU,1979-02-28,7,bourdais,French,\N,Sébastien,Bourdais
RAI,1979-10-17,8,raikkonen,Finnish,7,Kimi,Räikkönen
KUB,1984-12-07,9,kubica,Polish,88,Robert,Kubica
GLO,1982-03-18,10,glock,German,\N,Timo,Glock


In [0]:
%sql
select
nationality,
count(nationality) as total_driver
from sales.drivers_data
group by nationality
order by count(nationality) desc
Limit 5

nationality,total_driver
British,165
American,157
Italian,99
French,73
German,50
